In [ ]:
import time
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from dataclasses import dataclass


# ============================================================
# Problem Instance
# ============================================================

@dataclass
class CuttingStockInstance:
    stock_length: int
    items: list
    lengths: dict
    demands: dict

    @classmethod
    def from_csv(cls, stock_file="stock3.csv", items_file="items3.csv"):

        stock_df = pd.read_csv(stock_file)
        items_df = pd.read_csv(items_file)

        stock_length = int(stock_df.loc[0, "stock_length"])

        items = items_df["item"].tolist()
        lengths = dict(zip(items_df["item"], items_df["length"]))
        demands = dict(zip(items_df["item"], items_df["demand"]))

        return cls(stock_length, items, lengths, demands)


# ============================================================
# Column Generation Solver
# ============================================================

class CuttingStockDCG:

    def __init__(self, instance):
        self.data = instance

        self.master = gp.Model("CuttingStock_RMP")
        self.master.Params.OutputFlag = 1

        self.patterns = []   # list of dict patterns
        self.x = []          # master variables (one per pattern)

        self.demand_constr = {}

    # --------------------------------------------------------
    # Initial patterns (1-item greedy patterns)
    # --------------------------------------------------------
    def initialize_patterns(self):

        I = self.data.items
        L = self.data.stock_length

        for i in I:
            pattern = {j: 0 for j in I}
            pattern[i] = L // self.data.lengths[i]
            self.patterns.append(pattern)

    # --------------------------------------------------------
    # Randomized Greedy Patterns 
    # --------------------------------------------------------

    # --------------------------------------------------------
    # First-Fit Decreasing Patterns
    # --------------------------------------------------------
    def ffd_initialize_patterns(self, instance):
    
        L = instance.stock_length
    
        # sort items by decreasing size
        items_sorted = sorted(
            instance.items,
            key=lambda i: instance.lengths[i],
            reverse=True
        )
    
        # remaining demand copy (important!)
        remaining_demand = instance.demands.copy()
    
        patterns = []
    
        # while there is still demand left
        while sum(remaining_demand.values()) > 0:
    
            remaining_capacity = L
            pattern = {i: 0 for i in instance.items}
    
            # fill one roll greedily
            for i in items_sorted:
    
                if remaining_demand[i] <= 0:
                    continue
    
                max_fit = min(
                    remaining_demand[i],
                    remaining_capacity // instance.lengths[i]
                )
    
                if max_fit > 0:
                    pattern[i] = max_fit
                    remaining_demand[i] -= max_fit
                    remaining_capacity -= max_fit * instance.lengths[i]
    
            patterns.append(pattern)
    
        return patterns
    # --------------------------------------------------------
    # Build Restricted Master Problem (RMP)
    # --------------------------------------------------------
    def build_master(self):

        I = self.data.items

        for p, pattern in enumerate(self.patterns):

            col = gp.Column()

            for i in I:
                if pattern[i] > 0:
                    col.addTerms(pattern[i], self.demand_constr.get(i))

            var = self.master.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                obj=1,
                column=col,
                name=f"x_{p}"
            )

            self.x.append(var)

        # Demand constraints
        for i in I:
            self.demand_constr[i] = self.master.addConstr(
                gp.LinExpr() >= self.data.demands[i],
                name=f"demand_{i}"
            )

    # --------------------------------------------------------
    # Solve master problem
    # --------------------------------------------------------
    def solve_master(self):
        self.master.optimize()

    # --------------------------------------------------------
    # Pricing problem (Knapsack)
    # --------------------------------------------------------
    def pricing(self, duals):

        I = self.data.items
        L = self.data.stock_length

        pricing = gp.Model("pricing")
        pricing.Params.OutputFlag = 0

        y = pricing.addVars(I, vtype=GRB.INTEGER, lb=0, name="y")

        pricing.setObjective(
            gp.quicksum(duals[i] * y[i] for i in I),
            GRB.MAXIMIZE
        )

        pricing.addConstr(
            gp.quicksum(self.data.lengths[i] * y[i] for i in I)
            <= L
        )

        pricing.optimize()

        if pricing.Status != GRB.OPTIMAL:
            return None, 0

        reduced_cost = 1 - pricing.ObjVal

        if reduced_cost < -1e-6:
            pattern = {i: int(y[i].X) for i in I}
            return pattern, reduced_cost

        return None, reduced_cost

    # --------------------------------------------------------
    # Column Generation Loop
    # --------------------------------------------------------
    def solve(self):

        # Initialize patterns
        # self.initialize_patterns()
        self.patterns = self.ffd_initialize_patterns(self.data)

        # IMPORTANT: build constraints first (needed for columns)
        I = self.data.items
        for i in I:
            self.demand_constr[i] = self.master.addConstr(
                gp.LinExpr() >= self.data.demands[i],
                name=f"demand_{i}"
            )

        # now add initial columns
        for p, pattern in enumerate(self.patterns):

            col = gp.Column()
            for i in I:
                col.addTerms(pattern[i], self.demand_constr[i])

            var = self.master.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                obj=1,
                column=col,
                name=f"x_{p}"
            )

            self.x.append(var)

        self.master.update()

        iteration = 0

        while True:

            iteration += 1
            print(f"\nIteration {iteration}")

            self.solve_master()

            if self.master.Status != GRB.OPTIMAL:
                print("Master not optimal")
                break

            duals = {i: self.demand_constr[i].Pi for i in I}

            pattern, rc = self.pricing(duals)

            print("Reduced cost:", rc)

            if pattern is None:
                print("Optimal LP reached (no improving column).")
                break

            print("Adding pattern:", pattern)

            self.patterns.append(pattern)

            # build column
            col = gp.Column()
            for i in I:
                col.addTerms(pattern[i], self.demand_constr[i])

            var = self.master.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                obj=1,
                column=col,
                name=f"x_{len(self.x)}"
            )

            self.x.append(var)

            self.master.update()

        print("\nFinal LP objective:", self.master.ObjVal)


# ============================================================
# Main
# ============================================================

def main():

    start = time.perf_counter()

    instance = CuttingStockInstance.from_csv(
        "stock3.csv",
        "items3.csv"
    )

    solver = CuttingStockDCG(instance)
    solver.solve()

    print("\nTotal time:", round(time.perf_counter() - start, 4), "s")


if __name__ == "__main__":
    main()
